## STOR 893 Homework 1

Please see the homework assignment for details on what to implement. Starter code, including the 10 problems to test your code against, is included below.

In [1]:
import sys
import numpy as np
# Add S2MPJ and the python_problems subfolder to your path if it is not already configured
sys.path.append(r'C:\Users\woody\OneDrive\바탕 화면\FA26\STOR893\HW1\S2MPJ')
sys.path.append(r'C:\Users\woody\OneDrive\바탕 화면\FA26\STOR893\HW1\S2MPJ/python_problems')

In [2]:
#Basic gradient descent method using Armijo backtracking
def gradient_descent_backtracking(prob, x0, epsilon, c, maxit):
    x = x0
    f, g = prob.fgx(x)
    ng = np.linalg.norm(g)
    #Set initial stepsize to 1
    alpha = 1
    for i in range(maxit):
        #GD update
        xnew = x - alpha*g
        fnew = prob.fx(xnew)
        #Test Armijo Condition
        while(fnew > f - c*alpha*ng**2):
            #If we failed, reduce alpha
            alpha = alpha/2
            xnew = x - alpha*g
            fnew = prob.fx(xnew)
        #Double accepted alpha
        alpha = alpha*2
        #Update x and evaluate its gradient
        x = xnew
        f, g = prob.fgx(x)
        ng = np.linalg.norm(g)
        #If the gradient is sufficiently small, stop
        if(ng <= epsilon):
            return f, ng, i
    return f, ng, maxit

In [3]:
#List of problems
problist = ['ARWHEAD','BRYBND','ENGVAL1','EXTROSNB','FREUROTH','LIARWHD','NONDQUAR','TRIDIA','TQUARTIC','PENALTY1']
#These problems have variable dimension, setting all problems to n=100, to keep the problems small
paramlist = []
for i in range(len(problist)):
    paramlist.append('100')

In [5]:
import importlib

#Using importlib so we can use the problems as saved in problist
prob_index = 0
problem_name = problist[prob_index]

problem_module = importlib.import_module(problem_name)
problem_class = getattr(problem_module, problem_name)

#Initialize the problem as prob
prob = problem_class(paramlist[prob_index])
x0 = prob.x0
epsilon = 1e-4
c = 1e-3
maxit = 10000
x, its = gradient_descent_backtracking(prob, x0, epsilon, c, maxit)
#print(its)

print("iterations =", its)
print("final f =", prob.fx(x))
f, g = prob.fgx(x)
print("final gradient norm =", np.linalg.norm(g))

iterations = 164
final f = 1.8331136608651377e-10
final gradient norm = 8.932242851597545e-05


In [ ]:
#Some other useful commands:

# Evaluate function, gradient and Hessian at x
# f, g, H = prob.fgHx(x)

# Evaluate a Hessian vector product in the direction v at x (v must be a vector of the same size as x)
# Hxv = prob.fHxv(x, v)

### Add your code below

#### Cauchy TR

In [9]:
# Calculates the scalar in front of vector g in the Cauchy point step
def cauchy_step(ng, gHg, delta):
    if(gHg<=0):
        return -delta/ng
    else:
        return -min(ng**3/(delta*gHg), 1)*delta/ng

In [10]:
def tr_cauchy(prob, x0, delta0, eta, epsilon, maxit):
    x = x0
    delta = delta0

    n_comp = 0

    f, g = prob.fgx(x)
    Hxg = prob.fHxv(x, g)
    n_comp += 2

    ng = np.linalg.norm(g)
    gHg = (g.T @ Hxg).item()

    for i in range(maxit):

        # Converged before taking another iteration
        if ng <= epsilon:
            return f, ng, i, n_comp

        # Taking Cauchy step to approximate the subproblem
        p_coef = cauchy_step(ng, gHg, delta)
        p_norm = abs(p_coef)*ng
        x_new = x + p_coef*g
        f_new = prob.fx(x_new)

        m = f + p_coef*(ng**2) + 0.5*(p_coef**2)*gHg
        rho = (f - f_new)/(f - m)

        if rho > eta:
            x = x_new

            f, g = prob.fgx(x)
            Hxg = prob.fHxv(x, g)
            n_comp += 2

            ng = np.linalg.norm(g)
            gHg = (g.T @ Hxg).item()

            if np.isclose(delta, p_norm):
                delta = 2*delta

        else:
            delta = 0.5*delta
            n_comp += 1

        if (ng <= epsilon) or (n_comp >= maxit):
            return f, ng, i+1, n_comp

    return f, ng, maxit, n_comp






#### CG-Steihaug TR

In [4]:
# Calculate the TR subproblem using CG-Steihaug method. Returns p_k, H_k @ p_k, and the updated number of gradient evaluations / Hessian products
def cg_steihaug_step(prob, x, delta, cg_rho, g, n_comp):
    epsilon = cg_rho*np.linalg.norm(g)
    maxit_cg = g.size
    n = n_comp

    z = np.zeros_like(g)
    r = g.copy()
    d = -g

    if np.linalg.norm(r) <= epsilon:
        return z, z, n

    for i in range(maxit_cg):

        Hxd = prob.fHxv(x, d)
        n += 1
        dHd = (d.T @ Hxd).item()

        # Calculate candidates of tau
        a = np.linalg.norm(d) ** 2
        b = 2*(z.T @ d).item()
        quad_c = np.linalg.norm(z) ** 2 - delta**2
        disc = max(b**2 - 4*a*quad_c, 0.0)

        tau_n = (-b-np.sqrt(disc)) / (2*a)
        tau_p = (-b+np.sqrt(disc)) / (2*a)

        # Negative curvature
        if dHd <= 0:
            m_n = 0.5*dHd*(tau_n**2) + ((g.T @ d).item() + (z.T @ Hxd).item())*tau_n
            m_p = 0.5*dHd*(tau_p**2) + ((g.T @ d).item() + (z.T @ Hxd).item())*tau_p

            if m_n < m_p:
                return z + tau_n * d, r - g + tau_n * Hxd, n
            else:
                return z + tau_p * d, r - g + tau_p * Hxd, n

        # Regular CG step
        alpha = np.linalg.norm(r)**2 / dHd
        z_new = z + alpha*d

        # CG step goes outside trust region
        if np.linalg.norm(z_new) >= delta:
            return z + tau_p * d, r - g + tau_p * Hxd, n

        r_new = r + alpha*Hxd

        # CG convergence
        if np.linalg.norm(r_new) < epsilon:
            return z_new, r_new - g, n

        beta = np.linalg.norm(r_new)**2 / np.linalg.norm(r)**2
        d_new = -r_new + beta * d

        z = z_new
        r = r_new
        d = d_new

    return z, r - g, n


        
    

In [5]:
def tr_cg_steihaug(prob, x0, delta0, cg_rho, eta, epsilon, maxit):
    x = x0
    delta = delta0

    n_comp = 0

    f, g = prob.fgx(x)
    n_comp += 1

    for i in range(maxit):

        if np.linalg.norm(g) <= epsilon:
            return f, np.linalg.norm(g), i, n_comp

        p, Hxp, n_comp = cg_steihaug_step(prob, x, delta, cg_rho, g, n_comp)

        x_new = x + p

        rho = (f - prob.fx(x_new))/(- (g.T  @ p).item() - 0.5*(p.T @ Hxp).item())

        if rho > eta:
            x = x_new

            f, g = prob.fgx(x)
            n_comp += 1

            if np.isclose(delta, np.linalg.norm(p)):
                delta = 2 * delta

        else:
            delta = 0.5 * delta

        if (np.linalg.norm(g) <= epsilon) or (n_comp >= maxit):
            return f, np.linalg.norm(g), i+1, n_comp

    return f, np.linalg.norm(g), maxit, n_comp

            

### Run codes, get results

In [20]:
import importlib

#Using importlib so we can use the problems as saved in problist
prob_index = 0
problem_name = problist[prob_index]

problem_module = importlib.import_module(problem_name)
problem_class = getattr(problem_module, problem_name)

#Initialize the problem as prob
prob = problem_class(paramlist[prob_index])
x0 = prob.x0
delta0 = 1
eta = 0.25
epsilon = 1e-4
maxit = 10000



In [ ]:
f_fin, gn_fin, n_iter, n_comp = tr_cauchy(prob, x0, delta0, eta, epsilon, maxit)

print("Final f =", f_fin)
print("Final gradient norm =", gn_fin)
print("iterations:", n_iter)
print("number of gradient/Hessian product:", n_comp)

Final f = 1.6725321128063797e-10
Final gradient norm = 6.467200492979021e-05
iterations: 34
number of gradient/Hessian product: 70


In [38]:
f_fin, gn_fin, n_iter, n_comp = tr_cg_steihaug(prob, x0, delta0, 0.5, eta, epsilon, maxit)

print("Final f =", f_fin)
print("Final gradient norm =", gn_fin)
print("iterations:", n_iter)
print("number of gradient/Hessian product:", n_comp)

Final f = 0.0
Final gradient norm = 3.594691506384543e-08
iterations: 6
number of gradient/Hessian product: 15


#### THE REAL TEST!!!

In [39]:
#List of problems
problist = ['ARWHEAD','BRYBND','ENGVAL1','EXTROSNB','FREUROTH','LIARWHD','NONDQUAR','TRIDIA','TQUARTIC','PENALTY1']
#These problems have variable dimension, setting all problems to n=100, to keep the problems small
paramlist = []
for i in range(len(problist)):
    paramlist.append('100')

In [41]:
import importlib

#Using importlib so we can use the problems as saved in problist
for i in range(10):
    prob_index = i
    problem_name = problist[prob_index]

    problem_module = importlib.import_module(problem_name)
    problem_class = getattr(problem_module, problem_name)

    #Initialize the problem as prob
    prob = problem_class(paramlist[prob_index])
    x0 = prob.x0
    c = 1e-3
    epsilon = 1e-4
    maxit = 10000

    f, ng, its = gradient_descent_backtracking(prob, x0, epsilon, c, maxit)
    print("For problem", problem_name, ":")
    print(f"Function value: {f:.4e}, Gradient norm: {ng:.4e}, Iterations: {its:d}, Computations: {its:d}.")

    

For problem ARWHEAD :
Function value: 1.8331e-10, Gradient norm: 8.9322e-05, Iterations: 164, Computations: 164.
For problem BRYBND :
Function value: 4.1383e-10, Gradient norm: 7.0907e-05, Iterations: 289, Computations: 289.
For problem ENGVAL1 :
Function value: 1.0909e+02, Gradient norm: 7.3811e-05, Iterations: 42, Computations: 42.
For problem EXTROSNB :
Function value: 1.6650e-11, Gradient norm: 9.8650e-05, Iterations: 75, Computations: 75.
For problem FREUROTH :
Function value: 1.1965e+04, Gradient norm: 1.2341e-03, Iterations: 10000, Computations: 10000.
For problem LIARWHD :
Function value: 1.1378e-09, Gradient norm: 9.7695e-05, Iterations: 1930, Computations: 1930.
For problem NONDQUAR :
Function value: 1.0471e-04, Gradient norm: 2.4322e-04, Iterations: 10000, Computations: 10000.
For problem TRIDIA :
Function value: 1.6869e-09, Gradient norm: 9.0127e-05, Iterations: 5122, Computations: 5122.
For problem TQUARTIC :
Function value: 3.5920e-02, Gradient norm: 2.0527e-01, Iteration

In [11]:
import importlib

for i in range(10):
    prob_index = i
    problem_name = problist[prob_index]

    problem_module = importlib.import_module(problem_name)
    problem_class = getattr(problem_module, problem_name)

    #Initialize the problem as prob
    prob = problem_class(paramlist[prob_index])
    x0 = prob.x0
    delta0 = 1
    eta = 0.25
    epsilon = 1e-4
    maxit = 10000

    f, ng, its, comp = tr_cauchy(prob, x0, delta0, eta, epsilon, maxit)
    print("For problem", problem_name, ":")
    print(f"Function value: {f:.4e}, Gradient norm: {ng:.4e}, Iterations: {its:d}, Computations: {comp:d}.")

For problem ARWHEAD :
Function value: 1.6725e-10, Gradient norm: 6.4672e-05, Iterations: 34, Computations: 70.
For problem BRYBND :
Function value: 4.1978e-10, Gradient norm: 9.9896e-05, Iterations: 1040, Computations: 2079.
For problem ENGVAL1 :
Function value: 1.0909e+02, Gradient norm: 7.2015e-05, Iterations: 51, Computations: 104.
For problem EXTROSNB :
Function value: 9.3797e-03, Gradient norm: 3.6170e-02, Iterations: 4999, Computations: 10000.
For problem FREUROTH :
Function value: 1.1965e+04, Gradient norm: 3.2364e-03, Iterations: 4999, Computations: 10000.
For problem LIARWHD :
Function value: 1.3478e-09, Gradient norm: 7.8958e-05, Iterations: 52, Computations: 106.
For problem NONDQUAR :
Function value: 9.1271e-05, Gradient norm: 1.1177e-04, Iterations: 4999, Computations: 10000.
For problem TRIDIA :
Function value: 2.3149e-09, Gradient norm: 9.9767e-05, Iterations: 4932, Computations: 9866.
For problem TQUARTIC :
Function value: 1.6957e-02, Gradient norm: 3.1649e-02, Iteratio

In [7]:
import importlib

for i in range(10):
    prob_index = i
    problem_name = problist[prob_index]

    problem_module = importlib.import_module(problem_name)
    problem_class = getattr(problem_module, problem_name)

    #Initialize the problem as prob
    prob = problem_class(paramlist[prob_index])
    x0 = prob.x0
    delta0 = 1
    cg_rho = 0.5
    eta = 0.25
    epsilon = 1e-4
    maxit = 10000

    f, ng, its, comp = tr_cg_steihaug(prob, x0, delta0, cg_rho, eta, epsilon, maxit)
    print("For problem", problem_name, ":")
    print(f"Function value: {f:.4e}, Gradient norm: {ng:.4e}, Iterations: {its:d}, Computations: {comp:d}.")

For problem ARWHEAD :
Function value: 0.0000e+00, Gradient norm: 3.5947e-08, Iterations: 6, Computations: 15.
For problem BRYBND :
Function value: 2.8891e-11, Gradient norm: 6.5729e-05, Iterations: 18, Computations: 107.
For problem ENGVAL1 :
Function value: 1.0909e+02, Gradient norm: 9.8726e-05, Iterations: 16, Computations: 42.
For problem EXTROSNB :
Function value: 1.8448e-05, Gradient norm: 7.9803e-05, Iterations: 636, Computations: 2999.
For problem FREUROTH :
Function value: 1.1965e+04, Gradient norm: 8.1574e-05, Iterations: 53, Computations: 101.
For problem LIARWHD :
Function value: 1.4439e-14, Gradient norm: 4.2013e-06, Iterations: 16, Computations: 43.
For problem NONDQUAR :
Function value: 3.0762e-05, Gradient norm: 9.9550e-05, Iterations: 57, Computations: 491.
For problem TRIDIA :
Function value: 9.1463e-11, Gradient norm: 3.9221e-05, Iterations: 23, Computations: 198.
For problem TQUARTIC :
Function value: 3.1556e-08, Gradient norm: 3.7907e-05, Iterations: 20, Computation